# 4a · Train & tune

Trains and tunes the four models of the 2×2 design (holistic / atomic × BiLSTM / PubMedBERT) on train, selects on val, and saves each tuned model to `data/model/{model_name}/`. **The test set is not used here**: `4b_TestEval` loads the saved models and evaluates them.

**Inputs** (`data/`): `holistic_train_clean.csv`, `holistic_val_clean.csv`, `atomic_train.csv`, `atomic_val.csv` (notebook 2), `vocab.json`, `embed_matrix.npy` (notebook 3, built from train only).

**Outputs**
```
data/model/{model_name}/      one folder per tuned model (4 folders), read by 4b
    BiLSTM:     model.pt, vocab.json, model_info.json
    PubMedBERT: Hugging Face files (config.json, model.safetensors, tokenizer files), model_info.json
                model_info.json = architecture, config, label map, best hyper-parameters, val scores, history
outputs/figures/              training curves, val confusion matrices, Optuna plots
outputs/reports/              {model}_tuning_log.csv, {model}_optuna.db, best_hyperparameters.csv
outputs/checkpoints/          tuning scratch (best trial per model)
```

**Several sessions:** an experiment whose model folder is complete is skipped (`RETRAIN = False`) and the Optuna searches resume from their database, so a new session continues where the last one stopped. Sections 0 and 1 always run first.

**Shared code:** sections 0 and 1 (setup, datasets, model, trainers, configs, model folders) are also run by `4b_TestEval`, so the saved models are rebuilt with exactly these classes. Keep training-only code below the *End of the shared code* cell.

## Model settings (vs. the previous notebook)

**BiLSTM now matches Methodology §3.4.1**

| Item | Previous | Now (proposal) |
|---|---|---|
| Hidden units / direction | 300 | **128** (→ 256 combined) |
| Stacked BiLSTM layers | `nn.LSTM(num_layers=2)` | 2 stacked layers, **Dropout after each** layer |
| Dropout | 0.5 (+ embedding & pre-classifier dropout) | **0.3** after each BiLSTM layer and after the first Dense |
| Head | LayerNorm → 512 → 128 → 3 | **Dense 64 (ReLU) → Dense 3** (softmax inside `CrossEntropyLoss`) |
| Optimizer / LR | Adam, 3e-4 | **AdamW, 1e-3** |
| Tuning | none | **grid**: hidden {64,128,256} × lr {1e-3,5e-4,1e-4} × dropout {0.2,0.3,0.5}, best val Macro-F1 |

Kept on purpose: shared-weight (Siamese) encoder, BioWordVec 200-d, categorical cross-entropy (with the existing log-scaled class weights), masked mean-pooling, and the `[u, v, |u−v|, u·v]` pair-combination (a Siamese encoder needs *some* way to combine premise and hypothesis before the Dense layer).

**PubMedBERT now matches Methodology §3.4.2 and is tuned**

| Item | Previous | Now (proposal) |
|---|---|---|
| Weight decay | 0.01 | **0.001** |
| LR schedule | cosine, 10 % warm-up | **linear warm-up (10 %) → linear decay to 0** |
| Early stopping | patience 5 | **patience 2** on val Macro-F1, ≤ 5 epochs |
| Max sequence length | 256 (holistic) / 128 (atomic) | **512**, with *dynamic padding* so short batches stay cheap |
| Tuning | none (lr 2e-5, fixed batch) | **PubMedBERT** — Bayesian search with **Optuna** (TPE sampler + median pruner) instead of the 6-point grid. lr is searched continuously in 2e-5 … 5e-5, batch size in {16, 32}, plus weight decay, warm-up fraction and dropout. Selection metric: **val Macro-F1**. |

**How tuning plugs in.** Each `Tuner` keeps only the best trial's checkpoint (`outputs/checkpoints/{model}_best.pt`). `Experiment.run(tuned=tuner)` then reuses that checkpoint (no second training run) checks it on val and saves it to `data/model/`; the test set is never used for model selection. Set `TUNE_EPOCHS_*` below to shorten trials; the final model is then retrained once with the winning hyper-parameters for the full epoch budget.

## 0 · Setup

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, re, json, time
from dataclasses import dataclass, field, asdict
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, matthews_corrcoef,
)
from scipy.stats import chi2

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup,
)
from tqdm.auto import tqdm

DATA_DIR   = "data"
MODEL_DIR  = os.path.join(DATA_DIR, "model")          # one folder per tuned model: written by 4a, read by 4b
OUTPUT_DIR = "outputs"
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")  # tuning scratch (best trial per model)
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for _d in (DATA_DIR, MODEL_DIR, CKPT_DIR, FIG_DIR, REPORT_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_MAP = {"entailment": 0, "contradiction": 1, "neutral": 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
CLASSES   = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print(f"Device: {DEVICE}")
print(f"Classes: {CLASSES}")
print(f"\nData dir:    {DATA_DIR}/")
print(f"Models:      {MODEL_DIR}/")
print(f"Figures:     {FIG_DIR}/")
print(f"Reports:     {REPORT_DIR}/")

import copy, gc, itertools, functools
from dataclasses import replace

SEED = 592
def set_seed(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

# Optuna is only needed for the Bayesian BERT search in this notebook variant.
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna
print("optuna", optuna.__version__)

Device: cuda
Classes: ['entailment', 'contradiction', 'neutral']

Data dir:    data/
Models:      data\model/
Figures:     outputs\figures/
Reports:     outputs\reports/
optuna 5.0.0


## 1 · Shared code (also run by 4b_TestEval)

### 1.1 Label encoding

In [2]:
class LabelEncoder:
    """Converts a raw 'label' cell (int, numpy int, numeric string, or class
    name like 'neutral') into its integer class id. Try-int-first, fall back
    to the name→id map — this covers every label format seen across the CSVs.
    """
    def __init__(self, label_map: dict):
        self.label_map = label_map

    def encode(self, raw):
        try:
            return int(raw)
        except (ValueError, TypeError):
            return self.label_map[raw]

    def encode_series(self, series: pd.Series) -> pd.Series:
        return series.apply(self.encode)


LABEL_ENCODER = LabelEncoder(LABEL_MAP)

### 1.2 Datasets

`NLIPairDataset`for BioWordVec vocabulary BiLSTM. `BERTNLIDataset` HuggingFace tokenizer for PubMedBERT. Both route labels through `LabelEncoder`.

In [3]:
def tokenize(text, word2idx, max_len=128):
    tokens = re.findall(r"[a-z0-9']+", str(text).lower())[:max_len]
    ids = [word2idx.get(t, 1) for t in tokens]   # 1 = <UNK>
    return ids if ids else [1]


class NLIPairDataset(Dataset):
    """Premise/hypothesis pair dataset for the Siamese BiLSTM models.
    Used by both the holistic (3a) and atomic (3c) BiLSTM experiments —
    only `max_len` and the source CSVs differ between them.
    """
    def __init__(self, df, word2idx, label_encoder=LABEL_ENCODER, max_len=128):
        self.data          = df.reset_index(drop=True)
        self.word2idx      = word2idx
        self.label_encoder = label_encoder
        self.max_len       = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        p_ids = tokenize(row['premise'],    self.word2idx, self.max_len)
        h_ids = tokenize(row['hypothesis'], self.word2idx, self.max_len)
        label = self.label_encoder.encode(row['label'])
        return (torch.tensor(p_ids), torch.tensor(h_ids), torch.tensor(label))


def collate_fn(batch):
    p_seqs, h_seqs, labels = zip(*batch)
    p_lens = torch.tensor([len(s) for s in p_seqs])
    h_lens = torch.tensor([len(s) for s in h_seqs])
    p_pad  = pad_sequence(p_seqs, batch_first=True, padding_value=0)
    h_pad  = pad_sequence(h_seqs, batch_first=True, padding_value=0)
    return p_pad, p_lens, h_pad, h_lens, torch.stack(labels)


class BERTNLIDataset(Dataset):
    """Tokenises lazily WITHOUT padding; `bert_collate` pads each batch to its own longest
    sequence. This lets us honour the methodology's 512-token ceiling (max_len=512) without
    paying for 512-token padding on every example."""
    MIN_HYPOTHESIS_BUDGET = 32

    def __init__(self, df, tokenizer, label_encoder=LABEL_ENCODER, max_len=512):
        self.data          = df.reset_index(drop=True)
        self.tokenizer     = tokenizer
        self.label_encoder = label_encoder
        self.max_len       = max_len

    def __len__(self):
        return len(self.data)

    def _capped_premise(self, premise_text):
        max_premise_len = max(self.max_len - self.MIN_HYPOTHESIS_BUDGET, 1)
        premise_ids = self.tokenizer.encode(premise_text, add_special_tokens=False)
        if len(premise_ids) <= max_premise_len:
            return premise_text
        premise_ids = premise_ids[:max_premise_len]
        return self.tokenizer.decode(premise_ids)

    def __getitem__(self, idx):
        row     = self.data.iloc[idx]
        premise = self._capped_premise(str(row['premise']))
        enc = self.tokenizer(
            premise, str(row['hypothesis']),
            max_length=self.max_len, truncation='only_second',   # never truncate the (now-capped) premise
        )
        ids   = enc['input_ids']
        types = enc['token_type_ids'] if 'token_type_ids' in enc else [0] * len(ids)
        label = self.label_encoder.encode(row['label'])
        return {
            'input_ids':      torch.tensor(ids, dtype=torch.long),
            'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
            'token_type_ids': torch.tensor(types, dtype=torch.long),
            'label':          torch.tensor(label, dtype=torch.long),
        }


def bert_collate(batch, pad_id=0):
    """Dynamic padding: pad every tensor in the batch to the batch's longest sequence."""
    L = max(len(b['input_ids']) for b in batch)
    def pad(key, value):
        return torch.stack([F.pad(b[key], (0, L - len(b[key])), value=value) for b in batch])
    return {
        'input_ids':      pad('input_ids', pad_id),
        'attention_mask': pad('attention_mask', 0),
        'token_type_ids': pad('token_type_ids', 0),
        'label':          torch.stack([b['label'] for b in batch]),
    }

### 1.3 Model — Siamese BiLSTM

In [4]:
class SiameseBiLSTM(nn.Module):
    """Siamese BiLSTM NLI classifier — aligned with Methodology §3.4.1.

      BioWordVec embedding (200-d)
        -> BiLSTM layer 1 (H units / direction) -> Dropout
        -> BiLSTM layer 2 (H units / direction) -> Dropout        (stacked; context = 2H)
        -> masked mean-pool over time
      Premise and hypothesis go through the SAME encoder (shared weights), then are
      combined as [u, v, |u-v|, u*v] ->
        Dense(64, ReLU) -> Dropout -> Dense(3)   (softmax is applied inside CrossEntropyLoss;
                                                  use `predict_proba` for explicit probabilities)
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 pretrained_embeddings=None, freeze_embed=False, dropout=0.3, dense_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight = nn.Parameter(
                torch.tensor(pretrained_embeddings, dtype=torch.float32),
                requires_grad=not freeze_embed,
            )
        # Two explicitly stacked single-layer BiLSTMs so Dropout can follow EACH layer
        # (nn.LSTM(num_layers=2, dropout=p) only drops between layers, never after the last).
        self.lstm1 = nn.LSTM(embed_dim,      hidden_dim, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden_dim * 2, hidden_dim, batch_first=True, bidirectional=True)
        self.drop1   = nn.Dropout(dropout)
        self.drop2   = nn.Dropout(dropout)
        self.drop_fc = nn.Dropout(dropout)

        enc_dim = hidden_dim * 2                      # forward + backward = 256 when H = 128
        self.fc1 = nn.Linear(enc_dim * 4, dense_dim)  # Dense(64) + ReLU
        self.out = nn.Linear(dense_dim, n_classes)    # Dense(3)

    @staticmethod
    def _run(lstm, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
        out, _ = lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return out

    def encode(self, ids, lengths):
        x = self.embedding(ids)                                   # [B, L, 200]
        x = self.drop1(self._run(self.lstm1, x, lengths))         # [B, L, 2H]
        x = self.drop2(self._run(self.lstm2, x, lengths))         # [B, L, 2H]
        mask = (torch.arange(x.size(1), device=x.device)[None, :] < lengths.to(x.device)[:, None])
        mask = mask.float().unsqueeze(-1)
        return (x * mask).sum(1) / mask.sum(1).clamp(min=1)       # [B, 2H]

    def forward(self, prem_ids, prem_len, hyp_ids, hyp_len):
        u = self.encode(prem_ids, prem_len)
        v = self.encode(hyp_ids,  hyp_len)
        x = torch.cat([u, v, torch.abs(u - v), u * v], dim=-1)
        x = self.drop_fc(F.relu(self.fc1(x)))
        return self.out(x)

    @torch.no_grad()
    def predict_proba(self, prem_ids, prem_len, hyp_ids, hyp_len):
        return F.softmax(self.forward(prem_ids, prem_len, hyp_ids, hyp_len), dim=-1)

### 1.4 Class weights & plotting helpers

In [5]:
def compute_class_weights(df, n_classes, device, label_encoder=LABEL_ENCODER):
    """Inverse-frequency class weights with log-scaling to prevent extreme bias."""
    encoded = label_encoder.encode_series(df['label'])
    counts  = encoded.value_counts().sort_index()
    weights = 1.0 / counts.reindex(range(n_classes), fill_value=1).values
    weights = np.log(weights + 1.0)
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32).to(device)


def _safe_filename(name):
    """Filesystem-safe filename from an arbitrary plot title (handles
    em-dashes, ×, etc. that the old `title.replace(' ', '_')` left in place).
    """
    return re.sub(r"[^\w\-]+", "_", name).strip("_")


def plot_confusion(labels, preds, id2label, title, out_dir=FIG_DIR):
    cm    = confusion_matrix(labels, preds, labels=sorted(id2label))
    names = [id2label[i] for i in sorted(id2label)]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    out_path = os.path.join(out_dir, _safe_filename(title) + '.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved figure -> {out_path}")


def plot_training_curves(history, epochs, model_name, out_dir=FIG_DIR):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    er = range(1, len(history['train_loss']) + 1)   # early stopping can end before `epochs`
    ax1.plot(er, history['train_loss'], label='Train')
    ax1.plot(er, history['val_loss'],   label='Val')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='Loss'); ax1.legend()
    ax2.plot(er, history['val_acc'],      label='Accuracy')
    ax2.plot(er, history['val_macro_f1'], label='Macro-F1')
    ax2.set(title='Val Metrics', xlabel='Epoch', ylabel='Score'); ax2.legend()
    plt.suptitle(f'{model_name} - Training Curves', fontsize=13)
    plt.tight_layout()
    out_path = os.path.join(out_dir, f'{model_name}_curves.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved figure -> {out_path}")

### 1.5 Trainers

In [6]:
class BaseTrainer:
    def __init__(self, model, device, model_name, ckpt_dir=CKPT_DIR, patience=5):
        self.model      = model
        self.device     = device
        self.model_name = model_name
        self.patience   = patience
        self.history    = {"train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": []}
        self.best_macro = 0.0
        self.best_ckpt  = os.path.join(ckpt_dir, f"{model_name}_best.pt")

    def _step(self, batch):
        raise NotImplementedError

    def train_epoch(self, loader, optimizer, scheduler=None, epoch=None, epochs=None):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        desc = f"{self.model_name} train" + (f" [{epoch}/{epochs}]" if epoch else "")
        pbar = tqdm(loader, desc=desc, leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            logits, loss, labels = self._step(batch)
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
            pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.3f}")
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate(self, loader, id2label):
        self.model.eval()
        total_loss, all_preds, all_labels = 0.0, [], []
        for batch in tqdm(loader, desc=f"{self.model_name} eval", leave=False):
            logits, loss, labels = self._step(batch)
            total_loss  += loss.item() * len(labels)
            all_preds   += logits.argmax(1).cpu().tolist()
            all_labels  += labels.cpu().tolist()
        n      = len(all_labels)
        acc    = sum(p == l for p, l in zip(all_preds, all_labels)) / n
        macro  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        report = classification_report(
            all_labels, all_preds,
            labels=sorted(id2label), target_names=[id2label[i] for i in sorted(id2label)],
            zero_division=0,
        )
        return total_loss / n, acc, macro, report, all_preds, all_labels

    def fit(self, train_loader, val_loader, optimizer, epochs, id2label,
            scheduler=None, step_scheduler_every_batch=True,
            epoch_callback=None):
        batch_scheduler = scheduler if step_scheduler_every_batch else None
        patience_counter = 0

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = self.train_epoch(train_loader, optimizer, batch_scheduler,
                                                epoch=epoch, epochs=epochs)
            v_loss, v_acc, v_macro, _, _, _ = self.evaluate(val_loader, id2label)
            if not step_scheduler_every_batch and scheduler is not None:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(v_macro)
                else:
                    scheduler.step()

            self.history["train_loss"].append(tr_loss)
            self.history["val_loss"].append(v_loss)
            self.history["val_acc"].append(v_acc)
            self.history["val_macro_f1"].append(v_macro)

            if v_macro > self.best_macro:
                self.best_macro = v_macro
                torch.save(self.model.state_dict(), self.best_ckpt)
                flag = " * best"
                patience_counter = 0
            else:
                flag = ""
                patience_counter += 1

            print(f"Epoch {epoch:02d}/{epochs} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | "
                  f"val_loss={v_loss:.4f} val_acc={v_acc:.3f} val_F1={v_macro:.3f} | "
                  f"{time.time()-t0:.1f}s{flag}")

            if epoch_callback is not None:      # e.g. Optuna report + prune (may raise)
                epoch_callback(epoch, v_macro)

            if patience_counter >= self.patience:
                print(f"[EARLY STOPPING] Triggered after {epoch} epochs. Best val macro-F1: {self.best_macro:.4f}")
                break

        print(f"\nBest val macro-F1: {self.best_macro:.4f}")
        return self.history

    def load_best(self):
        self.model.load_state_dict(torch.load(self.best_ckpt, map_location=self.device))
        return self


class BiLSTMTrainer(BaseTrainer):
    def __init__(self, model, criterion, device, model_name, patience=5):
        super().__init__(model, device, model_name, patience=patience)
        self.criterion = criterion

    def _step(self, batch):
        p_ids, p_len, h_ids, h_len, labels = batch
        p_ids, p_len = p_ids.to(self.device), p_len.to(self.device)
        h_ids, h_len = h_ids.to(self.device), h_len.to(self.device)
        labels       = labels.to(self.device)
        logits = self.model(p_ids, p_len, h_ids, h_len)
        loss   = self.criterion(logits, labels)
        return logits, loss, labels


class BERTTrainer(BaseTrainer):
    def __init__(self, model, device, model_name, patience=5):
        super().__init__(model, device, model_name, patience=patience)

    def _step(self, batch):
        input_ids  = batch['input_ids'].to(self.device)
        attn_mask  = batch['attention_mask'].to(self.device)
        token_type = batch['token_type_ids'].to(self.device)
        labels     = batch['label'].to(self.device)
        out = self.model(input_ids=input_ids, attention_mask=attn_mask,
                          token_type_ids=token_type, labels=labels)
        return out.logits, out.loss, labels

### 1.6 Data loading & experiment configs

In [7]:
class BioWordVecVocab:
    def __init__(self, data_dir):
        with open(f"{data_dir}/vocab.json") as f:
            self.word2idx = json.load(f)
        self.embed_matrix = np.load(f"{data_dir}/embed_matrix.npy")
        self.vocab_size   = len(self.word2idx)


class NLIDataBundle:
    """Train and val of one granularity, as notebook 2 writes them. The test files are only read by 4b."""
    FILES = {
        "holistic": {"train": "holistic_train_clean.csv", "val": "holistic_val_clean.csv"},
        "atomic":   {"train": "atomic_train.csv",         "val": "atomic_val.csv"},
    }

    def __init__(self, data_dir, prefix):
        self.prefix   = prefix
        files         = self.FILES[prefix]
        self.train_df = self._load(os.path.join(data_dir, files["train"]))
        self.val_df   = self._load(os.path.join(data_dir, files["val"]))

    @staticmethod
    def _load(path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} not found: run notebook 2 first")
        df = pd.read_csv(path)
        unlabelled = df["label"].isna()
        if unlabelled.any():
            print(f"[WARNING] {path}: {int(unlabelled.sum()):,} rows without a label are left out")
        return df[~unlabelled].reset_index(drop=True)

    def summary(self):
        print(f"Train: {len(self.train_df)}  Val: {len(self.val_df)}")
        dist = pd.DataFrame({"train": self.train_df["label"].value_counts(),
                             "val": self.val_df["label"].value_counts()}).fillna(0).astype(int)
        print(dist.to_string())
        return self


@dataclass
class BiLSTMConfig:
    """Defaults follow Methodology §3.4.1 (BioWordVec 200-d, 2 stacked BiLSTM layers,
    128 hidden units/direction, Dropout 0.3, Dense 64 -> Dense 3, AdamW, lr 1e-3)."""
    model_name:   str
    data_prefix:  str
    max_len:      int
    embed_dim:    int   = 200
    hidden_dim:   int   = 128      # per direction -> 256 combined
    dense_dim:    int   = 64
    dropout:      float = 0.3
    batch_size:   int   = 64
    epochs:       int   = 15
    patience:     int   = 5
    lr:           float = 1e-3
    weight_decay: float = 0.01     # AdamW default; the proposal does not specify one
    n_classes:    int   = 3
    freeze_embed: bool  = False


@dataclass
class BERTConfig:
    """Defaults follow Methodology §3.4.2: max sequence 512, AdamW weight-decay 0.001,
    10 % linear warm-up then linear decay to 0, <=5 epochs, early stopping (patience 2)
    on val Macro-F1. `lr` and `batch_size` are the tuned hyper-parameters (grid:
    lr {2e-5, 3e-5, 5e-5} x batch {16, 32})."""
    model_name:     str
    data_prefix:    str
    bert_model:     str   = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
    max_len:        int   = 512
    batch_size:     int   = 16
    epochs:         int   = 5
    patience:       int   = 2
    lr:             float = 2e-5
    warmup_frac:    float = 0.1
    weight_decay:   float = 0.001
    hidden_dropout: float = 0.2
    n_classes:      int   = 3


def make_bilstm_components(cfg, vocab, train_df, encoder, device, epochs=None):
    """Model + loss + AdamW + cosine schedule. Shared by the tuner and the final experiment
    so a tuned configuration is built identically in both places."""
    epochs = epochs or cfg.epochs
    model = SiameseBiLSTM(
        vocab_size=vocab.vocab_size, embed_dim=cfg.embed_dim, hidden_dim=cfg.hidden_dim,
        n_classes=cfg.n_classes, pretrained_embeddings=vocab.embed_matrix,
        freeze_embed=cfg.freeze_embed, dropout=cfg.dropout, dense_dim=cfg.dense_dim,
    ).to(device)
    class_weights = compute_class_weights(train_df, cfg.n_classes, device, encoder)
    criterion     = nn.CrossEntropyLoss(weight=class_weights)
    optimizer     = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return model, criterion, optimizer, scheduler, class_weights


def make_bert_optim(model, cfg, steps_per_epoch, epochs=None):
    """AdamW + linear warm-up (first warmup_frac of steps) then linear decay to 0."""
    epochs       = epochs or cfg.epochs
    total_steps  = steps_per_epoch * epochs
    warmup_steps = int(total_steps * cfg.warmup_frac)
    optimizer    = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    return optimizer, scheduler, total_steps, warmup_steps

### 1.7 Model folders

The format of `data/model/{model_name}/`: written by 4a (`save_model_folder`), read by 4b (`load_model_folder`).

In [8]:
def model_path(model_name):
    return os.path.join(MODEL_DIR, model_name)


def _jsonable(o):
    return o.item() if hasattr(o, "item") else str(o)


def save_model_folder(config, model, info, word2idx=None, tokenizer=None):
    """Writes MODEL_DIR/{model_name}/:
         BiLSTM     -> model.pt (weights) + vocab.json (the word ids the model was trained with)
         PubMedBERT -> the Hugging Face files (config.json, model.safetensors, tokenizer files)
         both       -> model_info.json: architecture, config, label map, tuning result, val scores, history.
    model_info.json is written last: a folder without it is an unfinished save."""
    folder = model_path(config.model_name)
    os.makedirs(folder, exist_ok=True)
    info_path = os.path.join(folder, "model_info.json")
    if os.path.exists(info_path):
        os.remove(info_path)
    kind = "bilstm" if isinstance(config, BiLSTMConfig) else "bert"
    if kind == "bilstm":
        torch.save(model.state_dict(), os.path.join(folder, "model.pt"))
        with open(os.path.join(folder, "vocab.json"), "w") as f:
            json.dump(word2idx, f)
    else:
        model.save_pretrained(folder)
        tokenizer.save_pretrained(folder)
    info = {"kind": kind, "model_name": config.model_name, "data_prefix": config.data_prefix,
            "config": asdict(config), "label_map": LABEL_MAP, **info,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(info_path, "w") as f:
        json.dump(info, f, indent=2, default=_jsonable)
    print(f"Saved model -> {folder}/")
    return folder


def model_saved(model_name):
    return os.path.exists(os.path.join(model_path(model_name), "model_info.json"))


def load_model_folder(model_name, device=DEVICE):
    """Rebuilds a saved model for prediction. Returns (info, config, model, word2idx or tokenizer)."""
    folder = model_path(model_name)
    with open(os.path.join(folder, "model_info.json")) as f:
        info = json.load(f)
    if info["label_map"] != LABEL_MAP:
        raise ValueError(f"{folder}: trained with label map {info['label_map']}, this notebook uses {LABEL_MAP}")
    if info["kind"] == "bilstm":
        cfg = BiLSTMConfig(**info["config"])
        with open(os.path.join(folder, "vocab.json")) as f:
            word2idx = json.load(f)
        state = torch.load(os.path.join(folder, "model.pt"), map_location=device)
        model = SiameseBiLSTM(vocab_size=state["embedding.weight"].shape[0], embed_dim=cfg.embed_dim,
                              hidden_dim=cfg.hidden_dim, n_classes=cfg.n_classes, dropout=cfg.dropout,
                              dense_dim=cfg.dense_dim)
        model.load_state_dict(state)
        return info, cfg, model.to(device).eval(), word2idx
    cfg = BERTConfig(**info["config"])
    tokenizer = AutoTokenizer.from_pretrained(folder)
    model = AutoModelForSequenceClassification.from_pretrained(folder)
    return info, cfg, model.to(device).eval(), tokenizer

<!-- END OF SHARED CODE -->
---
**End of the shared code.** `4b_TestEval` runs every code cell above this one (sections 0 and 1). Everything below is training only.

## 2 · Training & tuning

### 2.1 Experiment classes

An experiment takes the tuner's best configuration, checks the final model on val and saves it to `data/model/`.

In [9]:
def _run_info(exp):
    """What 4a knows about a final model, stored in its model_info.json."""
    t = exp._tuned
    return {"best_params": t.best_params if t else None,
            "tuning_val_macro_f1": t.best_macro if t else None,
            "reused_tuning_checkpoint": exp._reuse,
            "val_accuracy": exp.val_acc, "val_macro_f1": exp.val_macro,
            "n_train": len(exp.bundle.train_df), "n_val": len(exp.bundle.val_df),
            "history": exp.history}


def _release_gpu(exp):
    """Frees the GPU after an experiment is saved: the model moves to the CPU, its gradients and the optimizer go.
    Otherwise every finished model stays on the GPU while the next experiment trains."""
    exp.model.zero_grad(set_to_none=True)
    exp.model.to("cpu")
    for attr in ("optimizer", "scheduler", "trainer", "criterion"):
        if hasattr(exp, attr):
            delattr(exp, attr)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return exp


class BiLSTMExperiment:
    """Final model of a BiLSTM experiment: adopts the tuner's best configuration (reusing its checkpoint when
    possible), checks it on val and saves it to MODEL_DIR/{model_name}/. The test set is not used (see 4b)."""
    def __init__(self, config: BiLSTMConfig, data_dir=DATA_DIR, device=DEVICE,
                 label_map=LABEL_MAP, id2label=ID2LABEL):
        self.config    = config
        self.data_dir  = data_dir
        self.device    = device
        self.id2label  = id2label
        self.encoder   = LabelEncoder(label_map)
        self._tuned    = None
        self._reuse    = False

    def apply_tuned(self, tuned):
        """Adopt the tuner's best configuration. If the tuner trained for the full epoch budget,
        its best checkpoint + history are reused (no retraining); otherwise we retrain once
        with the best hyper-parameters."""
        self._tuned = tuned
        self.config = tuned.best_config
        self._reuse = tuned.reusable
        print(f"Using tuned config for {self.config.model_name}: {tuned.best_params}  "
              f"(val macro-F1 {tuned.best_macro:.4f}; "
              f"{'reusing tuning checkpoint' if self._reuse else 'will retrain with best params'})")
        return self

    def load_data(self):
        self.bundle = NLIDataBundle(self.data_dir, self.config.data_prefix).summary()
        self.vocab  = BioWordVecVocab(self.data_dir)
        print(f"\nVocab: {self.vocab.vocab_size}   Embed matrix: {self.vocab.embed_matrix.shape}")
        return self

    def build_loaders(self):
        c = self.config
        mk_ds = lambda df: NLIPairDataset(df, self.vocab.word2idx, self.encoder, c.max_len)
        self.train_ds = mk_ds(self.bundle.train_df)
        self.val_ds   = mk_ds(self.bundle.val_df)

        mk_loader = lambda ds, shuffle: TorchDataLoader(
            ds, batch_size=c.batch_size, shuffle=shuffle, collate_fn=collate_fn)
        self.train_loader = mk_loader(self.train_ds, True)
        self.val_loader   = mk_loader(self.val_ds,   False)
        print(f"Train batches: {len(self.train_loader)}  Val batches: {len(self.val_loader)}")
        return self

    def build_model(self):
        c = self.config
        (self.model, self.criterion, self.optimizer,
         self.scheduler, class_weights) = make_bilstm_components(
            c, self.vocab, self.bundle.train_df, self.encoder, self.device)
        self.trainer = BiLSTMTrainer(self.model, self.criterion, self.device, c.model_name,
                                     patience=c.patience)
        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {total_params:,}")
        print(f"Class weights: {{ {', '.join(f'{self.id2label[i]}: {w:.3f}' for i, w in enumerate(class_weights.cpu()))} }}")
        return self

    def train(self):
        if self._reuse:
            self.history = self._tuned.best_history
            print("Skipping training — best tuning trial checkpoint is reused.")
            return self
        set_seed()
        self.history = self.trainer.fit(
            self.train_loader, self.val_loader, self.optimizer, self.config.epochs,
            self.id2label, scheduler=self.scheduler, step_scheduler_every_batch=False,
        )
        return self

    def plot_curves(self):
        plot_training_curves(self.history, self.config.epochs, self.config.model_name)
        return self

    def evaluate_val(self):
        self.trainer.load_best()
        _, self.val_acc, self.val_macro, report, preds, labels = \
            self.trainer.evaluate(self.val_loader, self.id2label)
        print(f"VAL  acc={self.val_acc:.4f}  macro-F1={self.val_macro:.4f}\n")
        print(report)
        plot_confusion(labels, preds, self.id2label, f"{self.config.model_name} - Val Confusion Matrix")
        return self

    def save(self):
        save_model_folder(self.config, self.model, _run_info(self), word2idx=self.vocab.word2idx)
        return self

    def release(self):
        return _release_gpu(self)

    def run(self, tuned=None):
        if tuned is not None:
            self.apply_tuned(tuned)
        return (self.load_data().build_loaders().build_model()
                    .train().plot_curves().evaluate_val().save().release())


class BERTExperiment:
    """Final model of a PubMedBERT experiment: same steps as BiLSTMExperiment."""
    def __init__(self, config: BERTConfig, data_dir=DATA_DIR, device=DEVICE,
                 label_map=LABEL_MAP, id2label=ID2LABEL):
        self.config    = config
        self.data_dir  = data_dir
        self.device    = device
        self.id2label  = id2label
        self.encoder   = LabelEncoder(label_map)
        self._tuned    = None
        self._reuse    = False

    def apply_tuned(self, tuned):
        self._tuned = tuned
        self.config = tuned.best_config
        self._reuse = tuned.reusable
        print(f"Using tuned config for {self.config.model_name}: {tuned.best_params}  "
              f"(val macro-F1 {tuned.best_macro:.4f}; "
              f"{'reusing tuning checkpoint' if self._reuse else 'will retrain with best params'})")
        return self

    def load_data(self):
        self.bundle = NLIDataBundle(self.data_dir, self.config.data_prefix).summary()
        return self

    def build_model(self):
        c = self.config
        self.tokenizer = AutoTokenizer.from_pretrained(c.bert_model)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            c.bert_model, num_labels=c.n_classes,
            id2label=self.id2label, label2id=LABEL_MAP,
            hidden_dropout_prob=c.hidden_dropout, attention_probs_dropout_prob=c.hidden_dropout,
        ).to(self.device)
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Loaded {c.bert_model}")
        print(f"Trainable parameters: {trainable:,}")
        return self

    def build_loaders(self):
        c = self.config
        collate = functools.partial(bert_collate, pad_id=self.tokenizer.pad_token_id or 0)
        mk_ds = lambda df: BERTNLIDataset(df, self.tokenizer, self.encoder, c.max_len)
        self.train_ds = mk_ds(self.bundle.train_df)
        self.val_ds   = mk_ds(self.bundle.val_df)

        mk_loader = lambda ds, shuffle: TorchDataLoader(
            ds, batch_size=c.batch_size, shuffle=shuffle, collate_fn=collate)
        self.train_loader = mk_loader(self.train_ds, True)
        self.val_loader   = mk_loader(self.val_ds,   False)
        print(f"Train batches: {len(self.train_loader)}   Val batches: {len(self.val_loader)}")

        self.optimizer, self.scheduler, total_steps, warmup_steps = make_bert_optim(
            self.model, c, len(self.train_loader))
        print(f"Total steps: {total_steps}   Warmup steps: {warmup_steps}")

        self.trainer = BERTTrainer(self.model, self.device, c.model_name, patience=c.patience)
        return self

    def train(self):
        if self._reuse:
            self.history = self._tuned.best_history
            print("Skipping training — best tuning trial checkpoint is reused.")
            return self
        set_seed()
        self.history = self.trainer.fit(
            self.train_loader, self.val_loader, self.optimizer, self.config.epochs,
            self.id2label, scheduler=self.scheduler, step_scheduler_every_batch=True,
        )
        return self

    def plot_curves(self):
        plot_training_curves(self.history, self.config.epochs, self.config.model_name)
        return self

    def evaluate_val(self):
        self.trainer.load_best()
        _, self.val_acc, self.val_macro, report, preds, labels = \
            self.trainer.evaluate(self.val_loader, self.id2label)
        print(f"VAL  acc={self.val_acc:.4f}  macro-F1={self.val_macro:.4f}\n")
        print(report)
        plot_confusion(labels, preds, self.id2label, f"{self.config.model_name} - Val Confusion Matrix")
        return self

    def save(self):
        save_model_folder(self.config, self.model, _run_info(self), tokenizer=self.tokenizer)
        return self

    def release(self):
        return _release_gpu(self)

    def run(self, tuned=None):
        if tuned is not None:
            self.apply_tuned(tuned)
        return (self.load_data().build_model().build_loaders()
                    .train().plot_curves().evaluate_val().save().release())

### 2.2 Hyperparameter tuning

Selection is always by **validation Macro-F1** (never the test set). Every trial is seeded identically (`SEED`), and only the best checkpoint is retained.

In [10]:
class TunerBase:
    """Bookkeeping shared by every tuner. Only the BEST trial's checkpoint is kept on disk
    (as outputs/checkpoints/{model_name}_best.pt) — a PubMedBERT checkpoint is ~440 MB, so
    keeping every trial would fill the disk fast."""
    reusable = True          # True if tuning used the full epoch budget -> no retraining needed

    def __init__(self, model_name):
        self.model_name   = model_name
        self.final_ckpt   = os.path.join(CKPT_DIR, f"{model_name}_best.pt")
        self.log_path     = os.path.join(REPORT_DIR, f"{model_name}_tuning_log.csv")
        self.best_macro   = -1.0
        self.best_params  = None
        self.best_history = None
        self.best_config  = None
        self.log          = []

    def _record(self, trial_id, params, score, epochs_run, status="complete"):
        self.log.append({"trial": trial_id, **params, "val_macro_f1": round(float(score), 4),
                         "epochs_run": epochs_run, "status": status})
        pd.DataFrame(self.log).to_csv(self.log_path, index=False)

    def _consider(self, trial_ckpt, score, params, history, config):
        """Promote this trial's checkpoint if it beats the best so far; otherwise delete it."""
        improved = os.path.exists(trial_ckpt) and score > self.best_macro
        if improved:
            os.replace(trial_ckpt, self.final_ckpt)
            self.best_macro, self.best_params = float(score), dict(params)
            self.best_history, self.best_config = history, config
        elif os.path.exists(trial_ckpt):
            os.remove(trial_ckpt)
        return improved

    @staticmethod
    def _free_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def summary(self, top=10):
        df = pd.DataFrame(self.log)
        if df.empty:
            print("No trials recorded."); return df
        df = df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
        print(f"\n{self.model_name}: {len(df)} trials  (log -> {self.log_path})")
        print(df.head(top).to_string(index=False))
        print(f"\nBEST  val macro-F1 = {self.best_macro:.4f}   params = {self.best_params}")
        return df


class BiLSTMGridSearch(TunerBase):
    """Exhaustive grid search for the BiLSTM (Methodology §3.4.1):
         hidden units {64, 128, 256}  x  learning rate {1e-3, 5e-4, 1e-4}  x  dropout {0.2, 0.3, 0.5}
       = 27 trials. The configuration with the highest validation Macro-F1 wins."""
    GRID = {"hidden_dim": [64, 128, 256], "lr": [1e-3, 5e-4, 1e-4], "dropout": [0.2, 0.3, 0.5]}

    def __init__(self, base_config: BiLSTMConfig, grid=None, tune_epochs=None,
                 data_dir=DATA_DIR, device=DEVICE):
        super().__init__(base_config.model_name)
        self.base        = base_config
        self.grid        = grid or self.GRID
        self.tune_epochs = tune_epochs
        self.reusable    = tune_epochs is None or tune_epochs == base_config.epochs
        self.device      = device
        self.encoder     = LabelEncoder(LABEL_MAP)
        self.bundle      = NLIDataBundle(data_dir, base_config.data_prefix)
        self.vocab       = BioWordVecVocab(data_dir)
        mk_ds = lambda df: NLIPairDataset(df, self.vocab.word2idx, self.encoder, base_config.max_len)
        self.train_ds, self.val_ds = mk_ds(self.bundle.train_df), mk_ds(self.bundle.val_df)

    def _trial(self, trial_id, params):
        cfg    = replace(self.base, **params)
        epochs = self.tune_epochs or cfg.epochs
        set_seed()
        model, criterion, optimizer, scheduler, _ = make_bilstm_components(
            cfg, self.vocab, self.bundle.train_df, self.encoder, self.device, epochs=epochs)
        train_loader = TorchDataLoader(self.train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate_fn)
        val_loader   = TorchDataLoader(self.val_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)
        trainer = BiLSTMTrainer(model, criterion, self.device,
                                f"{self.model_name}_gs{trial_id}", patience=cfg.patience)
        history = trainer.fit(train_loader, val_loader, optimizer, epochs, ID2LABEL,
                              scheduler=scheduler, step_scheduler_every_batch=False)
        score = trainer.best_macro
        self._consider(trainer.best_ckpt, score, params, history, cfg)
        n_epochs = len(history["val_loss"])
        del model, optimizer, scheduler, trainer
        self._free_memory()
        return score, n_epochs

    def run(self):
        keys, combos = list(self.grid), list(itertools.product(*self.grid.values()))
        print(f"BiLSTM grid search [{self.model_name}]: {len(combos)} trials "
              f"x up to {self.tune_epochs or self.base.epochs} epochs")
        for i, combo in enumerate(combos, 1):
            params = dict(zip(keys, combo))
            print(f"\n── trial {i}/{len(combos)}  {params}")
            score, n_epochs = self._trial(i, params)
            self._record(i, params, score, n_epochs)
        self.summary()
        return self

In [12]:
class BERTTuner(TunerBase):
    """Runs ONE PubMedBERT fine-tuning trial for a given hyper-parameter dict. Subclasses decide
    WHICH trials to run (grid vs. Optuna). Data, tokenizer and datasets are built once."""
    def __init__(self, base_config: BERTConfig, tune_epochs=None, data_dir=DATA_DIR, device=DEVICE):
        super().__init__(base_config.model_name)
        self.base        = base_config
        self.tune_epochs = tune_epochs
        self.reusable    = tune_epochs is None or tune_epochs == base_config.epochs
        self.device      = device
        self.encoder     = LabelEncoder(LABEL_MAP)
        self.bundle      = NLIDataBundle(data_dir, base_config.data_prefix)
        self.tokenizer   = AutoTokenizer.from_pretrained(base_config.bert_model)
        self.collate     = functools.partial(bert_collate, pad_id=self.tokenizer.pad_token_id or 0)
        mk_ds = lambda df: BERTNLIDataset(df, self.tokenizer, self.encoder, base_config.max_len)
        self.train_ds, self.val_ds = mk_ds(self.bundle.train_df), mk_ds(self.bundle.val_df)

    def _run_trial(self, trial_id, params, epoch_callback=None):
        cfg    = replace(self.base, **params)
        epochs = self.tune_epochs or cfg.epochs
        set_seed()
        model = AutoModelForSequenceClassification.from_pretrained(
            cfg.bert_model, num_labels=cfg.n_classes, id2label=ID2LABEL, label2id=LABEL_MAP,
            hidden_dropout_prob=cfg.hidden_dropout, attention_probs_dropout_prob=cfg.hidden_dropout,
        ).to(self.device)
        train_loader = TorchDataLoader(self.train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=self.collate)
        val_loader   = TorchDataLoader(self.val_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=self.collate)
        optimizer, scheduler, _, _ = make_bert_optim(model, cfg, len(train_loader), epochs)
        trainer = BERTTrainer(model, self.device, f"{self.model_name}_trial{trial_id}", patience=cfg.patience)
        try:
            history = trainer.fit(train_loader, val_loader, optimizer, epochs, ID2LABEL,
                                  scheduler=scheduler, step_scheduler_every_batch=True,
                                  epoch_callback=epoch_callback)
            score = trainer.best_macro
            self._consider(trainer.best_ckpt, score, params, history, cfg)
        finally:                                    # also runs when Optuna prunes the trial
            if os.path.exists(trainer.best_ckpt):
                os.remove(trainer.best_ckpt)
            del model, optimizer, scheduler, trainer, train_loader, val_loader
            self._free_memory()
        return score, history


class BERTOptunaSearch(BERTTuner):
    """Bayesian hyper-parameter search with Optuna (TPE sampler + MedianPruner) as an alternative
    to the methodology's 6-point grid. The search box is the SAME range as the grid for lr
    (2e-5 … 5e-5) and batch size {16, 32}, but lr is sampled continuously, and the regularisation
    knobs the grid keeps fixed (weight decay, warm-up fraction, dropout) are searched too.
    Poor trials are pruned after each epoch by comparing val Macro-F1 to the median of earlier trials."""

    def __init__(self, base_config, n_trials=12, timeout=None, space=None, resume=True,
                 seed=SEED, n_startup_trials=4, **kw):
        super().__init__(base_config, **kw)
        self.n_trials, self.timeout, self.resume, self.seed = n_trials, timeout, resume, seed
        self.n_startup_trials = n_startup_trials
        self.space = space or {
            "lr":             ("float_log", 2e-5, 5e-5),
            "batch_size":     ("cat",       [16, 32]),
            "weight_decay":   ("float_log", 1e-4, 1e-2),
            "warmup_frac":    ("float",     0.05, 0.20),
            "hidden_dropout": ("float",     0.10, 0.30),
        }
        self.study = None

    def _suggest(self, trial):
        p = {}
        for name, spec in self.space.items():
            kind = spec[0]
            if   kind == "float_log": p[name] = trial.suggest_float(name, spec[1], spec[2], log=True)
            elif kind == "float":     p[name] = trial.suggest_float(name, spec[1], spec[2])
            elif kind == "int":       p[name] = trial.suggest_int(name, spec[1], spec[2])
            elif kind == "cat":       p[name] = trial.suggest_categorical(name, spec[1])
            else: raise ValueError(f"unknown space kind {kind!r} for {name}")
        return p

    def _objective(self, trial):
        params = self._suggest(trial)
        last   = {"v": 0.0, "epochs": 0}

        def on_epoch(epoch, val_macro):
            last["v"], last["epochs"] = val_macro, epoch
            trial.report(val_macro, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        try:
            score, history = self._run_trial(trial.number, params, epoch_callback=on_epoch)
        except optuna.TrialPruned:
            self._record(trial.number, params, last["v"], last["epochs"], status="pruned")
            print(f"   ✂ pruned after epoch {last['epochs']} (val macro-F1 {last['v']:.4f})")
            raise
        trial.set_user_attr("history", history)         # lets a resumed study rebuild curves
        self._record(trial.number, params, score, len(history["val_loss"]))
        return score

    def run(self):
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        storage = f"sqlite:///{os.path.join(REPORT_DIR, self.model_name + '_optuna.db')}"
        if not self.resume:
            try: optuna.delete_study(study_name=self.model_name, storage=storage)
            except Exception: pass
        self.study = optuna.create_study(
            study_name=self.model_name, storage=storage, load_if_exists=True, direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=self.seed, n_startup_trials=self.n_startup_trials),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
        )
        done = [t for t in self.study.trials if t.state.is_finished()]
        if done:
            print(f"Resuming study '{self.model_name}': {len(done)} finished trials found.")
            complete = [t for t in done if t.state == optuna.trial.TrialState.COMPLETE]
            if complete and os.path.exists(self.final_ckpt):
                bt = self.study.best_trial
                self.best_macro, self.best_params = float(bt.value), dict(bt.params)
                self.best_history = bt.user_attrs.get("history")
                self.best_config  = replace(self.base, **bt.params)
        remaining = max(self.n_trials - len(done), 0)
        print(f"Optuna search [{self.model_name}]: running {remaining} more trial(s) "
              f"(target {self.n_trials}, up to {self.tune_epochs or self.base.epochs} epochs each)")
        if remaining:
            self.study.optimize(self._objective, n_trials=remaining, timeout=self.timeout,
                                gc_after_trial=True)
        if self.best_config is None:
            raise RuntimeError("No Optuna trial completed — nothing to reuse. Check the log above.")
        self.summary()
        self.plot_study()
        return self

    def plot_study(self):
        try:
            from optuna.visualization import matplotlib as ovm
            for fn, tag in ((ovm.plot_optimization_history, "history"),
                            (ovm.plot_intermediate_values,  "intermediate"),
                            (ovm.plot_param_importances,    "importance")):
                try:
                    fn(self.study)
                    plt.gcf().set_size_inches(7, 4)
                    plt.tight_layout()
                    plt.savefig(os.path.join(FIG_DIR, f"{self.model_name}_optuna_{tag}.png"), dpi=150)
                    plt.show()
                except Exception as e:            # e.g. importance needs >= 2 completed trials
                    print(f"(skipped optuna {tag} plot: {e})")
        except ImportError:
            print("optuna.visualization.matplotlib unavailable — skipping plots.")
        return self

In [13]:
# ── Tuning budget knobs ──────────────────────────────────────────────────────
# None  -> every trial uses the full epoch budget (BiLSTM: 15, BERT: 5) and the best trial's
#          checkpoint is reused as the final model (no retraining).
# int   -> shorter trials for a quick search; the winner is then retrained once for the full budget.
TUNE_EPOCHS_BILSTM = None
TUNE_EPOCHS_BERT   = None

# Optuna budget (per BERT experiment). 12-20 trials is a sensible range for a 2-D/5-D search
# with pruning; set higher if GPU time allows. Re-running a cell resumes the same study.
N_TRIALS_BERT = 12
TIMEOUT_BERT  = None      # seconds, or None

# False: an experiment whose model folder (data/model/{name}) is complete is skipped, so a new session can carry on.
# True : tune and train every experiment again (overwrites its model folder).
RETRAIN = False


def needs_training(model_name):
    if model_saved(model_name) and not RETRAIN:
        print(f"{model_name}: already saved in {model_path(model_name)}/ - skipped (RETRAIN = True to redo it)")
        return False
    return True

## 3 · Experiment 3a — Holistic BiLSTM

Siamese BiLSTM on whole answers vs. the premise. Grid search → best config → val check → saved to `data/model/holistic_bilstm/`.

In [ ]:
holistic_bilstm_config = BiLSTMConfig(
    model_name="holistic_bilstm",
    data_prefix="holistic",
    max_len=128,
)
if needs_training(holistic_bilstm_config.model_name):
    tuner_3a = BiLSTMGridSearch(holistic_bilstm_config, tune_epochs=TUNE_EPOCHS_BILSTM).run()
    exp_3a = BiLSTMExperiment(holistic_bilstm_config).run(tuned=tuner_3a)

## 4 · Experiment 3b — Holistic PubMedBERT

Fine-tunes BiomedNLP-BiomedBERT on premise–answer pairs (Optuna search → val check → saved to `data/model/holistic_pubmedbert/`). Truncation strategy is `only_second`: the premise is the evidence and is never cut. Max length 512 per the methodology; if you hit CUDA OOM at batch 32, lower `batch_size` in the search space.

In [ ]:
holistic_bert_config = BERTConfig(
    model_name="holistic_pubmedbert",
    data_prefix="holistic",
)
if needs_training(holistic_bert_config.model_name):
    tuner_3b = BERTOptunaSearch(holistic_bert_config, n_trials=N_TRIALS_BERT, timeout=TIMEOUT_BERT,
                                tune_epochs=TUNE_EPOCHS_BERT).run()
    exp_3b = BERTExperiment(holistic_bert_config).run(tuned=tuner_3b)

## 5 · Experiment 3c — Atomic BiLSTM

Same Siamese BiLSTM, trained on atoms (one claim each) instead of whole answers. Saved to `data/model/atomic_bilstm/`; 4b combines its atom predictions into one verdict per answer.

In [ ]:
atomic_bilstm_config = BiLSTMConfig(
    model_name="atomic_bilstm",
    data_prefix="atomic",
    max_len=64,   # atoms are ~10 words
)
if needs_training(atomic_bilstm_config.model_name):
    tuner_3c = BiLSTMGridSearch(atomic_bilstm_config, tune_epochs=TUNE_EPOCHS_BILSTM).run()
    exp_3c = BiLSTMExperiment(atomic_bilstm_config).run(tuned=tuner_3c)

## 6 · Experiment 3d — Atomic PubMedBERT

Fine-tunes PubMedBERT on the same atoms (Optuna search → val check → saved to `data/model/atomic_pubmedbert/`). Inputs are short, so dynamic padding keeps this much cheaper than the holistic model.

In [14]:
atomic_bert_config = BERTConfig(
    model_name="atomic_pubmedbert",
    data_prefix="atomic",
)
if needs_training(atomic_bert_config.model_name):
    tuner_3d = BERTOptunaSearch(atomic_bert_config, n_trials=N_TRIALS_BERT, timeout=TIMEOUT_BERT,
                                tune_epochs=TUNE_EPOCHS_BERT).run()
    exp_3d = BERTExperiment(atomic_bert_config).run(tuned=tuner_3d)

Resuming study 'atomic_pubmedbert': 3 finished trials found.
Optuna search [atomic_pubmedbert]: running 9 more trial(s) (target 12, up to 5 epochs each)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

atomic_pubmedbert_trial3 train [1/5]:   0%|          | 0/1150 [00:00<?, ?it/s]

atomic_pubmedbert_trial3 eval:   0%|          | 0/73 [00:00<?, ?it/s]

Epoch 01/5 | tr_loss=0.5660 tr_acc=0.736 | val_loss=0.2664 val_acc=0.901 val_F1=0.899 | 512.9s * best


... (5 outputs left out)

Epoch 03/5 | tr_loss=0.1448 tr_acc=0.949 | val_loss=0.2676 val_acc=0.916 val_F1=0.915 | 509.2s


atomic_pubmedbert_trial3 train [4/5]:   0%|          | 0/1150 [00:00<?, ?it/s]

[W 2026-11-09 09:24:08,061] Trial 3 failed with parameters: {'lr': 2.2484523367642915e-05, 'batch_size': 32, 'weight_decay': 0.00022952697014844923, 'warmup_frac': 0.11506317997340584, 'hidden_dropout': 0.24337301616864218} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\kynes\thesiskernel\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\kynes\AppData\Local\Temp\ipykernel_33004\3251926634.py", line 86, in _objective
    score, history = self._run_trial(trial.number, params, epoch_callback=on_epoch)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\kynes\AppData\Local\Temp\ipykernel_33004\3251926634.py", line 30, in _run_trial
    history = trainer.fit(train_loader, val_loader, optimizer, epochs, ID2LABEL,
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

## 7 · Saved models

Val scores and selected hyper-parameters of the four saved models (read from their `model_info.json`, so this also works in a new session).

In [ ]:
rows = []
for name in ["holistic_bilstm", "holistic_pubmedbert", "atomic_bilstm", "atomic_pubmedbert"]:
    if not model_saved(name):
        print(f"[MISSING] {name}: not saved yet")
        continue
    with open(os.path.join(model_path(name), "model_info.json")) as f:
        info = json.load(f)
    rows.append({"model": name, "val_macro_f1": round(info["val_macro_f1"], 4),
                 "val_accuracy": round(info["val_accuracy"], 4), "n_train": info["n_train"],
                 "epochs_run": len(info["history"]["val_loss"]) if info.get("history") else None,
                 "saved_at": info["saved_at"], **(info["best_params"] or {})})
best_hp = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
best_hp.to_csv(os.path.join(REPORT_DIR, "best_hyperparameters.csv"))
print(f"Saved -> {os.path.join(REPORT_DIR, 'best_hyperparameters.csv')}")
best_hp